<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/linear_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# 1. СОЗДАЕМ КЛАСС НАШЕЙ НЕЙРОСЕТИ
# В PyTorch любая модель оформляется как класс, наследуемый от nn.Module.
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        """
        Конструктор класса. Здесь мы объявляем слои, из которых состоит сеть.
        input_dim — количество входящих признаков (фичей).
        """
        super(LinearRegressionNet, self).__init__()

        # nn.Linear — это аналог Dense-слоя в Keras (линейный / полносвязный слой).
        # Первый аргумент (input_dim): сколько чисел принимает слой на вход.
        # Второй аргумент (1): сколько чисел возвращает на выходе (нам нужно 1 число-прогноз).
        # По умолчанию этот слой уже включает в себя вектор сдвига (bias).
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        """
        Этот метод описывает "прямой проход" (forward pass) данных через сеть.
        Здесь данные x проходят через наш единственный линейный слой.
        """
        # Передаем входные данные x в наш линейный слой и возвращаем результат
        return self.linear(x)

# 2. ИНИЦИАЛИЗИРУЕМ МОДЕЛЬ
# Предположим, у нас, как и в примере выше, 5 входных признаков.
# Создаем экземпляр нашего класса.
model = LinearRegressionNet(input_dim=5)

# 3. ОПРЕДЕЛЯЕМ ФУНКЦИЮ ПОТЕРЬ (КРИТЕРИЙ ОШИБКИ)
# nn.MSELoss() вычисляет среднеквадратичную ошибку между предсказанием сети и правильным ответом.
criterion = nn.MSELoss()

# 4. НАСТРАИВАЕМ ОПТИМИЗАТОР
# torch.optim.Adam — алгоритм оптимизации (вариант градиентного спуска).
# model.parameters() — передаем оптимизатору веса нашей модели, чтобы он знал, что именно нужно обновлять.
# lr=0.01 (Learning Rate) — шаг обучения. Определяет, насколько сильно веса будут меняться за один шаг.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ==========================================
# 1. АРХИТЕКТУРА МОДЕЛИ (уже знакомая вам)
# ==========================================
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionNet, self).__init__()
        self.linear = nn.Linear(input_dim, 1) # 2 входа (площадь, возраст) -> 1 выход (цена)

    def forward(self, x):
        return self.linear(x)


# ==========================================
# 2. ПОДГОТОВКА ДАННЫХ
# ==========================================
# Нам нужно предсказать цену квартиры. Пусть реальная формула будет:
# Цена = (Площадь * 3) - (Возраст * 0.5) + 10 + небольшой случайный шум

np.random.seed(42) # Фиксируем случайность для воспроизводимости

# Генерируем 100 случайных квартир:
# 1-й столбец: площадь (от 30 до 130 кв.м.)
# 2-й столбец: возраст дома (от 1 до 50 лет)
X_np = np.random.rand(100, 2)
X_np[:, 0] = X_np[:, 0] * 100 + 30
X_np[:, 1] = X_np[:, 1] * 50

# Считаем правильные ответы (цену в тыс. долларов) по нашей формуле с шумом
y_np = (X_np[:, 0] * 3) - (X_np[:, 1] * 0.5) + 10 + np.random.randn(100) * 2
y_np = y_np.reshape(-1, 1) # Превращаем в вектор-столбец

# ⚠️ ВАЖНО: PyTorch работает только со своими Тензорами (тип float32)
X_train = torch.tensor(X_np, dtype=torch.float32)
y_train = torch.tensor(y_np, dtype=torch.float32)


# ==========================================
# 3. ИНИЦИАЛИЗАЦИЯ И НАСТРОЙКА ОБУЧЕНИЯ
# ==========================================
model = LinearRegressionNet(input_dim=2) # У нас 2 признака
criterion = nn.MSELoss()                 # Функция потерь (среднеквадратичная ошибка)
optimizer = optim.Adam(model.parameters(), lr=0.1) # Оптимизатор с шагом обучения 0.1


# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ (TRAINING LOOP)
# ==========================================
epochs = 5000 # Сколько раз модель просмотрит все данные

for epoch in range(epochs):
    # Прямой проход: модель делает предсказание на основе текущих весов
    predictions = model(X_train)

    # Считаем ошибку (насколько модель промахнулась)
    loss = criterion(predictions, y_train)

    # Сбрасываем старые градиенты, чтобы они не копились
    optimizer.zero_grad()

    # Обратный проход: считаем, в какую сторону нужно изменить веса (производные)
    loss.backward()

    # Шаг оптимизатора: обновляем веса модели
    optimizer.step()

    # Выводим лог каждые 500 эпох, чтобы видеть прогресс
    if (epoch + 1) % 500 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}], Ошибка (Loss): {loss.item():.4f}")


# ==========================================
# 5. ИСПОЛЬЗОВАНИЕ МОДЕЛИ (ИНФЕРЕНС / ПРЕДСКАЗАНИЕ)
# ==========================================
# Переводим модель в режим оценки (выключает подсчет градиентов для экономии памяти)
model.eval()

print("\n--- Модель обучена! Делаем предсказание ---")

# У нас появилась новая квартира: Площадь = 85 кв.м., Возраст = 5 лет.
# Обязательно упаковываем данные в тензор такой же размерности, как при обучении (1 строка, 2 колонки)
new_apartment = torch.tensor([[85.0, 5.0]], dtype=torch.float32)

# Отключаем отслеживание градиентов, так как мы только предсказываем, а не учимся
with torch.no_grad():
    predicted_price = model(new_apartment)

# Извлекаем чистое число из тензора с помощью .item()
print(f"Новый объект: Площадь 85 кв.м, Возраст 5 лет.")
print(f"Предсказанная стоимость: ${predicted_price.item():.2f} тыс.")

# Наша истинная формула без шума дала бы: (85 * 3) - (5 * 0.5) + 10 = 255 - 2.5 + 10 = 262.5
